# D1-S2-Nb5: Normalisation
Showcase:
- state-verification:
    - measuring starting volume is sufficient using pipettes
- auto-correction / self-healing aPs
    - plate offset

## Set your `protocol_mode`

...so it is easy to switch between simulation and execution

In [1]:
prep_simulation = False  # True or False

run_identifier = "nb5"

Provide directory to your input/design file, and the normalisation targets:

In [2]:
design_file_directory = "./_design_templates/d1_s2_nb5_design_example.csv"

target_concentration_ng_uL = 10  # every sample ends at this concentration
sample_volume_uL = 10  # the same sample volume from every well; lowered only when it must be
max_fill = 0.9  # fill an output well to at most 90% of what it holds

### Design file
One row per sample: which plate and well it is in, what it is, its concentration, and how much of it
there is to dilute.

In [3]:
from _helpers.nb4_helper import load_design_file

design_file_df = load_design_file(design_file_directory)
design_file_df

,plate_id,well_id,sample_name,sample_type,concentration_ng_uL,available sample to dilute (uL)
0,plate_1,A1,Sample 1,DNA,55,24
1,plate_1,B1,Sample 2,DNA,25,24
2,plate_1,C1,Sample 3,DNA,68,24
3,plate_1,D1,Sample 4,DNA,12,24
4,plate_1,E1,Sample 5,DNA,18,24
5,plate_1,F1,Sample 6,DNA,95,24
6,plate_1,G1,Sample 7,DNA,18,24
7,plate_1,H1,Sample 8,DNA,55,24
8,plate_1,A2,Sample 9,DNA,120,24
9,plate_1,B2,Sample 10,DNA,12,24


## Library Import

In [4]:
import asyncio
import logging
import time

---
## Visualizer3D
...so everything that follows has one highest-level reference frame

In [5]:
from pylabrobot.resources import Coordinate, Resource
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

facility = Facility(name="facility", size_x=2_400, size_y=1_000, size_z=0)

viewer = Viewer3D(root = facility, name="nb5_normalisation")

await viewer.start()

viewer on http://127.0.0.1:1339/#token=sm9dNwzdVfb_e7sd-QKJ9sL9LpAkqNwRtbn4wODdPRE  (websocket 2123)


viewer: a browser connected, drawing with WebGPU


2026-09-28 08:12:51,561 - pylabrobot.visualizer3D.server - WARNING - refused a websocket without this viewer's token: a page left open from an earlier viewer, or one opened without the link it printed. Repeats are logged at debug.


---

## Set your `logging`
We (and our agents) always benefit from a log.

This follows standard Python logging but for convenience we set up:
- showing logging inside notebook using `verbose` declaration
- simple way to log to a file (in append mode)

I recommend always logging all commands you send/receive from any device for auditing purposes.

In [6]:
import datetime
import logging
import time

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

# one log folder per mode
prep_mode = "simulation" if prep_simulation else "execution"
protocol_mode = f"prep_{prep_mode}"

started_at = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"_logs/{protocol_mode}/{run_identifier}_{started_at}.log"
pylabrobot.setup_logger(log_path, level=LOG_LEVEL_IO)

pylabrobot.verbose(True, level=logging.INFO)

print(f"logging to {log_path}")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

2026-09-28 08:12:50,870 - pylabrobot - INFO - --- nb5 (prep_execution) ---


logging to _logs/prep_execution/nb5_20260928-081250.log


## Device setups

### Prep

In [7]:
from pylabrobot.hamilton.prep import Prep
from pylabrobot.hamilton.prep.driver.simulator import RECORDING_PREP_HEATER_SHAKER

HOST = "192.168.100.102"  # as in the Prep demo notebook; ignored in simulation
PORT = 2000

prep = Prep(
    simulation=prep_simulation,
    declared_configuration_json=RECORDING_PREP_HEATER_SHAKER,  # our Prep: 8-channel head, heater shaker
    host=HOST,
    port=PORT,
    # deck_components_offset=Coordinate(1.1,1,0)
    )

facility.assign_child_resource(
    prep,
    location=Coordinate(x=0.0, y=0.0, z=0.0)
)

prep.update_deck_components(offset=Coordinate(1.1, 1.0, 0)
                            )
await prep.setup()
await prep.lights.turn_on()

2026-09-28 08:12:51,250 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Initializing Hamilton connection...
2026-09-28 08:12:51,701 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Registering Hamilton client...
2026-09-28 08:12:51,704 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton root objects...
2026-09-28 08:12:51,706 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton global objects...
2026-09-28 08:12:51,710 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client setup complete. Client ID: 5, globals: 1
2026-09-28 08:12:51,798 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Initializing Hamilton connection...
2026-09-28 08:12:52,297 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Registering Hamilton client...
2026-09-28 08:12:52,300 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton root objects...
2026-09-28 08:12:52,301 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton global objects

---

## Adaptive process planning

### Volumes per sample
Every sample gives the same volume, `sample_volume_uL`; the diluent is what varies:
`diluent_uL = sample_uL × (concentration / target − 1)`. Every sample ends at the same
concentration, in its own volume (`final_uL`).

The sample volume is lowered only when it must be: when less is available, or when the diluted
sample would not fit in the well (`sample_adapted`).

In [8]:
from pylabrobot.resources.biorad import biorad_96_wellplate_200uL_Vb

from _helpers.nb4_helper import compute_dilutions

# the most an output well may hold: 90% of its volume, from the plate's own definition
well = biorad_96_wellplate_200uL_Vb(name="well")["A1"][0]
max_well_volume_uL = max_fill * well.compute_volume_from_height(well.get_size_z())

design_file_df = compute_dilutions(
    design_file_df, target_concentration_ng_uL, sample_volume_uL, max_well_volume_uL
)
design_file_df

,plate_id,well_id,sample_name,sample_type,concentration_ng_uL,available sample to dilute (uL),sample_uL,diluent_uL,final_uL,status
0,plate_1,A1,Sample 1,DNA,55,24,10,45.0,55.0,normalise
1,plate_1,B1,Sample 2,DNA,25,24,10,15.0,25.0,normalise
2,plate_1,C1,Sample 3,DNA,68,24,10,58.0,68.0,normalise
3,plate_1,D1,Sample 4,DNA,12,24,10,2.0,12.0,normalise
4,plate_1,E1,Sample 5,DNA,18,24,10,8.0,18.0,normalise
5,plate_1,F1,Sample 6,DNA,95,24,10,85.0,95.0,normalise
6,plate_1,G1,Sample 7,DNA,18,24,10,8.0,18.0,normalise
7,plate_1,H1,Sample 8,DNA,55,24,10,45.0,55.0,normalise
8,plate_1,A2,Sample 9,DNA,120,24,10,110.0,120.0,normalise
9,plate_1,B2,Sample 10,DNA,12,24,10,2.0,12.0,normalise


---
## Deck setup

we are now starting to build more complex setups and need more resources:

In [9]:
from pylabrobot.resources import set_tip_tracking, set_volume_tracking

set_tip_tracking(enabled=True)
set_volume_tracking(enabled=True)

In [10]:
import math

from pylabrobot.resources import ContainerRack, ResourceHolder, create_ordered_items_2d
from pylabrobot.resources.biorad import biorad_96_wellplate_200uL_Vb
from pylabrobot.resources.hamilton import (
    Trough_CAR_5R60_A00,
    hamilton_1_trough_60mL_Vb,
    hamilton_96_tiprack_50uL_NTR,
    hamilton_96_tiprack_300uL_NTR,
    hamilton_plate_carrier_L5_ac,
    hamilton_tip_carrier_L5_ntr_a00,
)

In [11]:
from _helpers.nb4_helper import hamilton_5_troughrack_60mL

### `diluent`
The water trough: `prep_diluent`.

In [12]:
# Prep: the 60 mL trough in position 0 of the trough rack
prep_troughrack = prep.deck[0] = hamilton_5_troughrack_60mL(name="prep_troughrack")
prep_troughrack[0] = prep_diluent = hamilton_1_trough_60mL_Vb(name="prep_diluent")

# what is in the trough at the start, in uL
prep_diluent.tracker.set_volume(40_000)

### Sample plates
Each device has 2 dedicated positions for sample plates, one per `plate_id` in the design file.
Each sample's well starts with the volume the design file says is available.

In [13]:
from _helpers.nb4_helper import new_plates

plate_ids = list(design_file_df["plate_id"].unique())
n_plates = len(plate_ids)
assert 1 <= n_plates <= 2, f"the design file has {n_plates} plates; each device has 2 plate positions"
# a 50 uL tip per sample and 2 for the diluent: 2 racks, 192 tips, serve at most 190
assert len(design_file_df) <= 190, f"{len(design_file_df)} samples; at most 190 fit the 50 uL tips"

In [14]:
# Prep: deck sites 4, 5, right of the diluent's trough rack (site 0)
prep_plate_sites = [5,3 ]

prep_sample_plates = new_plates("prep", design_file_df)
for site, plate in zip(prep_plate_sites, prep_sample_plates):
    prep.deck[site] = plate

In [15]:
# Prep: deck sites 2, 3 for the normalisation plates, empty to start
prep_output_sites = [4, 2]

prep_output_plates = [biorad_96_wellplate_200uL_Vb(name=f"prep_{plate_id}_output") for plate_id in plate_ids]
for site, plate in zip(prep_output_sites, prep_output_plates):
    prep.deck[site] = plate

then create new tipracks and assign them to the deck:

In [16]:
prep_tips_50uL_00 = prep.deck[1] = hamilton_96_tiprack_50uL_NTR(name="prep_tips_50uL_00")
prep_tips_50uL_01 = prep.deck[6] = hamilton_96_tiprack_50uL_NTR(name="prep_tips_50uL_01")

prep_tips_300uL_00 = prep.deck[7] = hamilton_96_tiprack_300uL_NTR(name="prep_tips_300uL_00")

---
## Normalisation on the Prep
Its two independent channels each take their own volume, so every sample gets its own diluent.
Samples go in pairs, one per channel, plate by plate.

In [17]:
channels = [0, 1]
clld = prep.pipettes.LLDMode.CAPACITIVE  # every aspirate finds the liquid surface itself
input_plates = dict(zip(plate_ids, prep_sample_plates))
output_plates = dict(zip(plate_ids, prep_output_plates))
# 50 uL tips, taken in order by both steps
tips_50uL = iter(prep_tips_50uL_00.get_all_items() + prep_tips_50uL_01.get_all_items())


def pairs_of(samples):
    """The samples two at a time, one per channel, plate by plate."""
    pairs = []
    for _, plate_samples in samples.groupby("plate_id", sort=False):
        rows = plate_samples.to_dict("records")
        pairs += [rows[i : i + 2] for i in range(0, len(rows), 2)]
    return pairs

### Step 0.3: is there enough water?
Before anything moves, probe the water trough. The run needs at most a full output well per sample,
plus what the trough keeps back, plus 5%. Too little, and the operator tops it up first.

In [18]:
output_well_uL = 200  # a Bio-Rad HSP9601 well holds 200 uL
dead_volume_uL = 500  # what the trough keeps back
tolerance = 0.05

water_needed_uL = (output_well_uL * len(design_file_df) + dead_volume_uL) * (1 + tolerance)
print(f"water needed: {water_needed_uL:.0f} uL")

water needed: 5565 uL


In [19]:
async def probe_water_uL():
    """The water in the trough: both channels, 3 probes each, averaged."""
    volumes = await prep.pipettes.probe_liquid_volumes(
        containers=[prep_diluent, prep_diluent], use_channels=channels, lld_mode=clld, n_replicates=3
    )
    return sum(volumes) / len(volumes)


if not prep_simulation:  # a simulated trough holds no liquid to find
    await prep.pipettes.pick_up_tips(tip_spots=prep_tips_300uL_00["C1:D1"], use_channels=channels)

    water_uL = await probe_water_uL()
    while water_uL < water_needed_uL:
        missing_mL = (water_needed_uL - water_uL) / 1000
        answer = input(
            f"Measured {water_uL / 1000:.2f} mL of water. Add at least {missing_mL:.2f} mL, "
            "then press Enter (or type 'skip' to run anyway): "
        )
        if answer.strip().lower() == "skip":
            break
        water_uL = await probe_water_uL()

    print(f"water: {water_uL:.0f} uL measured, {water_needed_uL:.0f} uL needed")
    await prep.pipettes.discard_tips(use_channels=channels)

water: 51111 uL measured, 5565 uL needed


### Step 1: diluent into the empty normalisation plates
The tip follows the volume: up to 50 uL with 50 uL tips (their water classes blow out), above that
with 300 uL tips (a jet). One pair of tips for each: the diluent only ever goes into empty wells.

In [20]:
async def add_diluent(samples, tip_spots, jet, blow_out):
    """Diluent into the empty output wells, two at a time, with one pair of tips."""
    if samples.empty:
        return
    await prep.pipettes.pick_up_tips(tip_spots=tip_spots, use_channels=channels)

    for pair in pairs_of(samples):
        use = channels[: len(pair)]
        wells_out = [output_plates[row["plate_id"]][row["well_id"]][0] for row in pair]
        volumes = [row["diluent_uL"] for row in pair]
        liquid_class = dict(jet=[jet] * len(pair), blow_out=[blow_out] * len(pair))

        await prep.pipettes.aspirate(
            containers=[prep_diluent] * len(pair),
            volumes=volumes,
            use_channels=use,
            lld_mode=clld,
            **liquid_class,
        )
        await prep.pipettes.dispense(containers=wells_out, volumes=volumes, use_channels=use, **liquid_class)

    await prep.pipettes.discard_tips(use_channels=channels)


diluent = design_file_df["diluent_uL"]
small = design_file_df[(diluent > 0) & (diluent <= 50)]  # too dilute (0 uL) gets none
large = design_file_df[diluent > 50]

In [21]:
await add_diluent(small, [next(tips_50uL), next(tips_50uL)], jet=False, blow_out=True)
await add_diluent(large, prep_tips_300uL_00["A1:B1"], jet=True, blow_out=False)

2026-09-28 08:14:07,693 - pylabrobot.hamilton.prep.driver.features.pipettes - WARNING - channel 0 measured 51282.2 uL in prep_diluent where the model had 40000.0 uL


### Step 2: the sample on top, mixed
Fresh 50 uL tips for every pair. The 50 uL tips' water liquid classes all blow out, so these calls ask
for `blow_out`.

In [23]:
await prep.pipettes.return_tips()

In [ ]:
for pair in pairs_of(design_file_df):
    use = channels[: len(pair)]
    wells_in = [input_plates[row["plate_id"]][row["well_id"]][0] for row in pair]
    wells_out = [output_plates[row["plate_id"]][row["well_id"]][0] for row in pair]
    volumes = [row["sample_uL"] for row in pair]
    mix_volumes = [min(row["final_uL"] / 2, 40) for row in pair]
    blow_out = [True] * len(pair)

    await prep.pipettes.pick_up_tips(tip_spots=[next(tips_50uL) for _ in pair], use_channels=use)

    await prep.pipettes.aspirate(
        containers=wells_in, volumes=volumes, use_channels=use, lld_mode=clld, blow_out=blow_out
    )
    await prep.pipettes.dispense(containers=wells_out, volumes=volumes, use_channels=use, blow_out=blow_out)

    for cycle in range(3):  # mix
        await prep.pipettes.aspirate(
            containers=wells_out, volumes=mix_volumes, use_channels=use, blow_out=blow_out
        )
        await prep.pipettes.dispense(containers=wells_out, volumes=mix_volumes, use_channels=use, blow_out=blow_out)

    await prep.pipettes.discard_tips(use_channels=use)

2026-09-28 08:24:21,793 - pylabrobot.hamilton.prep.driver.features.pipettes - WARNING - channel 0 measured 111.0 uL in prep_plate_1_input_well_C1 where the model had 24.0 uL
2026-09-28 08:24:21,794 - pylabrobot.hamilton.prep.driver.features.pipettes - WARNING - channel 1 measured 100.8 uL in prep_plate_1_input_well_D1 where the model had 24.0 uL
2026-09-28 08:25:55,977 - pylabrobot.hamilton.prep.driver.features.pipettes - WARNING - channel 0 measured 112.3 uL in prep_plate_1_input_well_E1 where the model had 24.0 uL
2026-09-28 08:25:55,978 - pylabrobot.hamilton.prep.driver.features.pipettes - WARNING - channel 1 measured 101.4 uL in prep_plate_1_input_well_F1 where the model had 24.0 uL


### Record
The design file, with what each sample was given and its status.

In [ ]:
import os

os.makedirs("_results", exist_ok=True)
results_path = f"_results/{run_identifier}_{started_at}_normalisation.csv"
design_file_df.to_csv(results_path, index=False)

print(f"results written to {results_path}")

---
# Shutdown

In [8]:
await prep.stop()

In [ ]:
await viewer.stop()